# CS 3170 — Concurrency and Coroutines
**Live coding notebook** · James Anderson · Mobile Application Development · Wright State University

Adapted from **10_Concurrency.pptx** (19 slides), following the same sequence: weather examples → structured concurrency and errors → RaceTracker → cancellation.

**How to teach with this notebook**
- Run the cells in order with **Shift+Enter**. Code cells share definitions and variables.
- In the **Live** edition, search for `TODO (live)` for places to type code during class. These cells remain syntactically valid before the exercise, but the output may not yet demonstrate the intended behavior.
- The **Completed** edition contains the corresponding working versions and practice solutions.
- A notebook runs **regular JVM Kotlin**, not the Android/Compose runtime. RaceTracker is simulated in the console; Compose snippets near the end are **Android Studio examples**, shown as non-executable code.

**First run:** Kotlin Notebook may need to download the coroutines dependency from Maven Central. The dependency cell is intentionally separate from the import cell.

Android course reference: https://developer.android.com/courses/android-basics-compose/unit-5


## 0 · Setup
Load Kotlin coroutines for JVM. **Run this cell first.**


In [2]:
@file:DependsOn("org.jetbrains.kotlinx:kotlinx-coroutines-core-jvm:1.10.2")


In [3]:
import kotlinx.coroutines.*
import kotlin.system.measureTimeMillis


## 1 · What is concurrency? (slide 2)

- **Synchronous:** one operation finishes before the next one starts.
- **Asynchronous:** a task can start and allow other work to continue without waiting for it to finish.
- **Coroutine:** a unit of work whose execution may **suspend** and later **resume**.
- In an Android app, this helps keep the UI responsive while fetching data, saving data, or loading resources.

**Discussion:** Concurrency does *not* necessarily mean parallel execution on multiple CPU threads.


## 2 · Synchronous code and `runBlocking` (slide 3)
First, make a *blocking* one-second weather demonstration. Notice that `Thread.sleep` blocks the current thread.


In [4]:
println("Weather forecast")
// TODO (live): Make this function wait one second using Thread.sleep(...).
println("Sunny")


Weather forecast
Sunny


### Suspending instead of sleeping
`delay()` is a **suspending function**, so it must run in a coroutine or another suspending function. `runBlocking` is useful for this **notebook demonstration and tests**, but do **not** use it to block Android's main/UI thread.

**Try it:** Uncomment a call to `delay(1000)` *outside* a coroutine in a fresh cell. What compiler error do you get?


In [5]:
runBlocking {
    println("Weather forecast")
    // TODO (live): suspend for one second with delay(...).
    println("Sunny")
}


Weather forecast
Sunny


## 3 · Suspending functions (slide 4)
These two functions simulate network calls. Mark them `suspend` so they can call `delay`. A **suspension point** is a place where a coroutine may pause and later continue.


In [6]:
suspend fun printForecast() {
    // TODO (live): Add a one-second delay before printing the forecast.
    println("Sunny")
}

suspend fun printTemperature() {
    // TODO (live): Add a one-second delay before printing the temperature.
    println("30°C")
}


In [7]:
runBlocking {
    println("Weather forecast")
    printForecast()
    printTemperature()
}


Weather forecast
Sunny
30°C


## 4 · Time the sequential version (slide 5)
`measureTimeMillis` measures elapsed milliseconds. With two one-second calls **in sequence**, we expect roughly **2,000 ms** (plus overhead).


In [8]:
val sequentialMs = measureTimeMillis {
    runBlocking {
        println("Weather forecast")
        printForecast()
        printTemperature()
    }
}
println("Sequential: ${sequentialMs} ms")


Weather forecast
Sunny
30°C
Sequential: 0 ms


## 5 · `launch`: concurrent tasks (slide 6)
`launch` starts a child coroutine and returns immediately with a `Job`. The enclosing `runBlocking` waits for its child coroutines before returning.

**Predict:** Will `"Launched both requests"` print before the weather results? How long will the two one-second requests take together?


In [9]:
val launchedMs = measureTimeMillis {
    runBlocking {
        println("Weather forecast")
        // TODO (live): Start printForecast() and printTemperature()
        // in separate launch { ... } blocks.
        println("Launched both requests")
    }
}
println("With launch: ${launchedMs} ms")


Weather forecast
Launched both requests
With launch: 1 ms


**Discuss:** `launch` is for a task whose **result value you do not need**. If you need a value back, use `async` and `await` instead.


## 6 · `async`, `Deferred`, and `await` (slide 7)
For the next version, make functions **return strings** rather than print them. `async` returns `Deferred<T>`, and `await()` supplies its result (suspending if necessary).


In [10]:
suspend fun getForecast(): String {
    // TODO (live): Simulate one second of network latency.
    return "Sunny"
}

suspend fun getTemperature(): String {
    // TODO (live): Simulate one second of network latency.
    return "30°C"
}


In [11]:
val asyncMs = measureTimeMillis {
    runBlocking {
        // TODO (live): Start two async { ... } calls returning String.
        val forecast: Deferred<String> = async { getForecast() }
        val temperature: Deferred<String> = async { getTemperature() }
        // TODO (live): Use await() to combine both results.
        println("${forecast.await()} ${temperature.await()}")
    }
}
println("With async: ${asyncMs} ms")


Sunny 30°C
With async: 7 ms


## 7 · Parallel decomposition and `coroutineScope` (slide 8)
Break the weather report into independent subtasks, then combine their results. `coroutineScope` **waits for its children to finish** before returning; callers see one suspending function that gives them the complete report.


In [12]:
suspend fun getWeatherReport(): String = coroutineScope {
    val forecast = async { getForecast() }
    val temperature = async { getTemperature() }
    "${forecast.await()} ${temperature.await()}"
}

runBlocking {
    println("Complete report: ${getWeatherReport()}")
}


Complete report: Sunny 30°C


### Your turn
Edit the cell below so the two operations run concurrently and return a **single string**. In the Live version, the cell is a safe place to start typing.


In [13]:
// TODO (live): Write getWeatherReportWithLabel() with coroutineScope,
// async, and await. Then call it with runBlocking.


## 8 · Exceptions in child coroutines (slide 9)
Child coroutines form a **parent–child hierarchy**. By default, an unhandled exception in one child **cancels the surrounding scope and its siblings**. Catch the failure at the appropriate boundary.


In [14]:
suspend fun getBrokenWeatherReport(): String = coroutineScope {
    val forecast = async { getForecast() }
    val temperature = async<String> {
        delay(200)
        throw IllegalStateException("Temperature service is unavailable")
    }
    "${forecast.await()} ${temperature.await()}"
}

runBlocking {
    try {
        println(getBrokenWeatherReport())
    } catch (e: IllegalStateException) {
        println("Caught error: ${e.message}")
        println("Weather report unavailable right now")
    }
}


Caught error: Temperature service is unavailable
Weather report unavailable right now


## 9 · `try` / `catch` and a fallback (slide 10)
If the error is **expected and recoverable**, catch it **inside the temperature task** so the rest of the report can still succeed.


In [15]:
suspend fun getWeatherReportWithFallback(): String = coroutineScope {
    val forecast = async { getForecast() }
    val temperature = async {
        try {
            delay(200)
            throw IllegalStateException("Temperature service is unavailable")
        } catch (e: IllegalStateException) {
            println("Handled temperature error: ${e.message}")
            "temperature unavailable"
        }
    }
    "${forecast.await()}; ${temperature.await()}"
}

runBlocking { println(getWeatherReportWithFallback()) }


Handled temperature error: Temperature service is unavailable
Sunny; temperature unavailable


**Important:** `CancellationException` is different from an ordinary recoverable error. If you catch it for cleanup or logging, **rethrow it** so cancellation is not swallowed.


## 10 · RaceTracker, without an Android UI (slides 11–12)
The slides switch to **RaceTracker**, where two players make progress concurrently. A Kotlin notebook cannot run a Compose UI, so this version prints progress to the cell output.

We will give each player a `currentProgress`, `maxProgress`, `progressIncrement`, and delay between increments.


In [16]:
class RaceParticipant(
    val name: String,
    private val progressIncrement: Int,
    private val progressDelayMillis: Long
) {
    var currentProgress: Int = 0
        private set
    val maxProgress: Int = 100

    suspend fun run() {
        try {
            while (currentProgress < maxProgress) {
                // TODO (live): delay for progressDelayMillis milliseconds.
                currentProgress = (currentProgress + progressIncrement)
                    .coerceAtMost(maxProgress)
                println("$name: $currentProgress%")
            }
            println("$name finished!")
        } catch (e: CancellationException) {
            println("$name was cancelled at $currentProgress%")
            throw e
        }
    }
}


## 11 · Start two runners together (slides 13–16)
`launch` begins each race; `coroutineScope` waits until **both** finish. Only then should `raceInProgress` become `false`.

**Compare with the slides:** setting `raceInProgress = false` immediately after two `launch` calls may remove a `LaunchedEffect` from composition and cancel its child work. The scope is what makes the sequencing correct.


In [17]:
runBlocking {
    val playerOne = RaceParticipant("Player 1", progressIncrement = 20, progressDelayMillis = 200)
    val playerTwo = RaceParticipant("Player 2", progressIncrement = 25, progressDelayMillis = 300)
    var raceInProgress = true

    coroutineScope {
        // TODO (live): launch both playerOne.run() and playerTwo.run().
    }
    raceInProgress = false
    println("Race in progress? $raceInProgress")
}


Race in progress? false


## 12 · Compose version: `LaunchedEffect` (slides 13–16)
**Android Studio reference — do not execute this snippet in the notebook.** The actual `RaceTracker` app has Compose state and progress bars.

The problematic version in the slides starts child coroutines and **immediately** sets `raceInProgress = false`:

```kotlin
if (raceInProgress) {
    LaunchedEffect(playerOne, playerTwo) {
        launch { playerOne.run() }
        launch { playerTwo.run() }
        raceInProgress = false  // Too soon: launch is non-blocking.
    }
}
```

Wrap the two launches in a `coroutineScope` so that `false` is assigned only **after both finish**:

```kotlin
import androidx.compose.runtime.LaunchedEffect
import kotlinx.coroutines.coroutineScope
import kotlinx.coroutines.launch

if (raceInProgress) {
    LaunchedEffect(playerOne, playerTwo) {
        coroutineScope {
            launch { playerOne.run() }
            launch { playerTwo.run() }
        }
        raceInProgress = false
    }
}
```

In Compose, entering the conditional starts the effect; removing it from composition cancels it. This is how **Pause / Reset** can stop the work. When resetting, make sure app state and the players' progress are reset separately.


## 13 · Cancellation and Reset (slides 17–18)
Cancellation is **cooperative**: a suspending call such as `delay()` observes cancellation and throws `CancellationException`. For resource cleanup or a log message, catch it and **rethrow** it, as in `RaceParticipant.run()` above.


In [18]:
runBlocking {
    val runner = RaceParticipant("Demo runner", progressIncrement = 10, progressDelayMillis = 150)
    val job: Job = launch { runner.run() }

    delay(420)
    println("Reset clicked")
    // TODO (live): cancel job and wait for it to finish.
    // Hint: job.cancelAndJoin()
    println("Progress now: ${runner.currentProgress}%")
}


Demo runner: 10%
Demo runner: 20%
Demo runner: 30%
Demo runner: 40%
Demo runner: 50%
Demo runner: 60%
Demo runner: 70%
Demo runner: 80%
Demo runner: 90%
Demo runner: 100%
Demo runner finished!
Reset clicked
Progress now: 100%


## 14 · `Job`: active, cancelled, and completed (slide 19)
`launch` returns a `Job`. Use `isActive`, `isCancelled`, `isCompleted`, or `cancel()` / `cancelAndJoin()` to manage it. A job tracks parent–child relationships.


In [19]:
runBlocking {
    val job = launch {
        repeat(6) { step ->
            delay(150)
            println("Working... step ${step + 1}")
        }
    }
    println("Just launched: active=${job.isActive}, completed=${job.isCompleted}")
    delay(350)
    job.cancelAndJoin()
    println("After cancellation: cancelled=${job.isCancelled}, completed=${job.isCompleted}")
}


Just launched: active=true, completed=false
Working... step 1
Working... step 2
After cancellation: cancelled=true, completed=true


## 15 · Quick challenges / exit ticket
1. Change the delays to 500 ms each: what do **sequential** and **concurrent** execution times become?
2. Change one runner's speed. Which player wins? Can the slower one still show progress before the faster one finishes?
3. What happens if a sibling coroutine throws an exception during a race?
4. Why is `coroutineScope` needed before setting `raceInProgress = false`?

Type your experiment in the cell below.


In [20]:
// TODO (live): Try a variation of the examples above.
// Keep this in a runBlocking { ... } block if you call suspend functions.


---
### Recap
- `delay()` **suspends** a coroutine; `Thread.sleep()` **blocks** a thread.
- `launch` starts work and returns a `Job`; `async` starts work and returns `Deferred<T>`.
- `await()` waits for a result; `coroutineScope` waits for its children.
- By default, child failure propagates through the scope; cancellation should remain cooperative.
- **Android:** use lifecycle-aware coroutine APIs such as `LaunchedEffect` for composables, **not** `runBlocking` on the UI thread.

Original slide reference: [Android Basics with Compose — Unit 5](https://developer.android.com/courses/android-basics-compose/unit-5).
